In [1]:
from pathlib import Path
import pandas as pd

data_dir = Path("../datos")

etiquetas = pd.read_csv(data_dir / "etiquetas_train.csv")
combos = pd.read_csv(data_dir / "combos_a_predecir.csv")

facturas_encabezado         = pd.read_parquet(data_dir / "oinv.parquet")
facturas_detalle            = pd.read_parquet(data_dir / "inv1.parquet")
notas_credito_encabezado    = pd.read_parquet(data_dir / "orin.parquet")
notas_credito_detalle       = pd.read_parquet(data_dir / "rin1.parquet")
catalogo_productos          = pd.read_parquet(data_dir / "oitm.parquet")
grupo_productos             = pd.read_parquet(data_dir / "oitb.parquet")
inventario_diario           = pd.read_parquet(data_dir / "inventario_regional.parquet")
ubicaciones                 = pd.read_parquet(data_dir / "obin.parquet")
bodegas                     = pd.read_parquet(data_dir / "owhs.parquet")

## A1. SQL. Escribe en ventas.sql (SQL Server) la consulta que devuelve la venta neta de agosto por línea de documento: facturas y notas de débito en positivo, notas de crédito en negativo, con SKU, grupo de artículo, bodega, canal y ubicación. Luego reprodúcela en Python. No hay plantilla: diseña tú los joins y los filtros.

Se unen los detalles con sus encabezados y se conservan los documentos no cancelados del 1 al 21 de agosto. Las facturas y notas de débito llevan signo positivo; las notas de crédito, negativo. El signo se aplica al importe convertido, sin modificar LineTotal.

El SKU proviene de InvntryUom y el canal de la ubicación. Se conserva una fila por ItemCode en el catálogo, priorizando el segmento específico, para evitar multiplicar ventas. Los joins de catálogo mantienen las líneas aunque falten referencias.

Las facturas en USD se convierten con DocRate. Se consideran las notas de crédito en GTQ según la moneda de la empresa indicada en el enunciado. StockValue y GrssProfit se conservan como datos originales y no intervienen en la venta neta.


/* ventas.sql */

```sql
USE slc_prueba;

;WITH productos_priorizados AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY ItemCode
            ORDER BY
                CASE
                    WHEN U_Segmento IS NULL
                      OR U_Segmento = 'SIN SEGMENTO'
                    THEN 1
                    ELSE 0
                END,
                U_Segmento
        ) AS prioridad
    FROM dbo.oitm
),
documentos AS (
    SELECT
        'FACTURA' AS TipoDocumento,
        detalle.DocEntry,
        detalle.LineNum,
        encabezado.DocDate,
        encabezado.DocSubType,
        detalle.ItemCode,
        detalle.WhsCode,
        detalle.U_Ubicacion,
        encabezado.DocCur,
        encabezado.DocRate,
        1 AS Signo,
        detalle.Quantity,
        detalle.LineTotal,
        detalle.StockValue,
        detalle.GrssProfit
    FROM dbo.inv1 AS detalle
    INNER JOIN dbo.oinv AS encabezado
        ON detalle.DocEntry = encabezado.DocEntry
    WHERE encabezado.Canceled = 'N'
      AND encabezado.DocDate >= '20260801'
      AND encabezado.DocDate < '20260822'

    UNION ALL

    SELECT
        'NOTA_CREDITO' AS TipoDocumento,
        detalle.DocEntry,
        detalle.LineNum,
        encabezado.DocDate,
        NULL AS DocSubType,
        detalle.ItemCode,
        detalle.WhsCode,
        detalle.U_Ubicacion,
        NULL AS DocCur,
        NULL AS DocRate,
        -1 AS Signo,
        detalle.Quantity,
        detalle.LineTotal,
        detalle.StockValue,
        detalle.GrssProfit
    FROM dbo.rin1 AS detalle
    INNER JOIN dbo.orin AS encabezado
        ON detalle.DocEntry = encabezado.DocEntry
    WHERE encabezado.Canceled = 'N'
      AND encabezado.DocDate >= '20260801'
      AND encabezado.DocDate < '20260822'
)
SELECT
    documento.TipoDocumento,
    documento.DocEntry,
    documento.LineNum,
    documento.DocDate,
    documento.DocSubType,
    documento.ItemCode,
    producto.InvntryUom AS SKU,
    grupo.GrupoArticulo,
    documento.WhsCode,
    bodega.WhsName,
    ubicacion.Canal,
    ubicacion.Ubicacion,
    documento.DocCur,
    documento.DocRate,
    documento.Signo,
    documento.Quantity,
    documento.LineTotal,
    CASE
        WHEN documento.DocCur = 'USD'
            THEN documento.LineTotal * documento.DocRate
        WHEN documento.DocCur = 'QTZ'
            THEN documento.LineTotal
        WHEN documento.TipoDocumento = 'NOTA_CREDITO'
            THEN documento.LineTotal
        ELSE NULL
    END AS ImporteGTQ,
    documento.StockValue,
    documento.GrssProfit,
    SUM(
    documento.Signo *
    CASE
        WHEN documento.DocCur = 'USD'
            THEN documento.LineTotal * documento.DocRate
        WHEN documento.DocCur = 'QTZ'
            THEN documento.LineTotal
        WHEN documento.TipoDocumento = 'NOTA_CREDITO'
            THEN documento.LineTotal
        ELSE NULL
    END
) OVER () AS VentaNetaTotalGTQ
FROM documentos AS documento
LEFT JOIN productos_priorizados AS producto
    ON documento.ItemCode = producto.ItemCode
   AND producto.prioridad = 1
LEFT JOIN dbo.oitb AS grupo
    ON producto.ItmsGrpCod = grupo.ItmsGrpCod
LEFT JOIN dbo.owhs AS bodega
    ON documento.WhsCode = bodega.WhsCode
LEFT JOIN dbo.obin AS ubicacion
    ON documento.U_Ubicacion = ubicacion.AbsEntry;
```

In [2]:
catalogo_priorizado = (
    catalogo_productos.assign(
        prioridad_segmento=(
            catalogo_productos["U_Segmento"].isna()
            | catalogo_productos["U_Segmento"].eq("SIN SEGMENTO")
        ).astype(int)
    )
    .sort_values(
        ["ItemCode", "prioridad_segmento", "U_Segmento"],
        na_position="last"
    )
    .drop_duplicates("ItemCode", keep="first")
    .drop(columns="prioridad_segmento")
)

assert catalogo_priorizado["ItemCode"].is_unique

In [3]:
ventas_facturas = facturas_detalle.merge(
    facturas_encabezado,
    on="DocEntry",
    how="inner",
    validate="many_to_one"
)

ventas_facturas = ventas_facturas.loc[
    ventas_facturas["Canceled"].eq("N")
    & ventas_facturas["DocDate"].ge("2026-08-01")
    & ventas_facturas["DocDate"].lt("2026-08-22")
].copy()

ventas_facturas["TipoDocumento"] = "FACTURA"
ventas_facturas["Signo"] = 1


In [4]:
ventas_credito = notas_credito_detalle.merge(
    notas_credito_encabezado,
    on="DocEntry",
    how="inner",
    validate="many_to_one"
)

ventas_credito = ventas_credito.loc[
    ventas_credito["Canceled"].eq("N")
    & ventas_credito["DocDate"].ge("2026-08-01")
    & ventas_credito["DocDate"].lt("2026-08-22")
].copy()

ventas_credito["TipoDocumento"] = "NOTA_CREDITO"
ventas_credito["Signo"] = -1
ventas_credito["DocSubType"] = None
ventas_credito["DocCur"] = None
ventas_credito["DocRate"] = float("nan")


In [5]:
columnas = [
    "TipoDocumento", "DocEntry", "LineNum", "DocDate", "DocSubType",
    "ItemCode", "WhsCode", "U_Ubicacion", "DocCur", "DocRate",
    "Signo", "Quantity", "LineTotal", "StockValue", "GrssProfit",
]

documentos = pd.concat(
    [ventas_facturas[columnas], ventas_credito[columnas]],
    ignore_index=True
)

In [6]:
ventas = (
    documentos
    .merge(
        catalogo_priorizado[["ItemCode", "InvntryUom", "ItmsGrpCod"]],
        on="ItemCode",
        how="left",
        validate="many_to_one"
    )
    .rename(columns={"InvntryUom": "SKU"})
    .merge(
        grupo_productos,
        on="ItmsGrpCod",
        how="left",
        validate="many_to_one"
    )
    .merge(
        bodegas[["WhsCode", "WhsName"]],
        on="WhsCode",
        how="left",
        validate="many_to_one"
    )
    .merge(
        ubicaciones[["AbsEntry", "Ubicacion", "Canal"]],
        left_on="U_Ubicacion",
        right_on="AbsEntry",
        how="left",
        validate="many_to_one"
    )
    .drop(columns="AbsEntry")
)

In [7]:
ventas["ImporteGTQ"] = float("nan")

moneda_local = ventas["DocCur"].eq("QTZ")
moneda_usd = ventas["DocCur"].eq("USD")
es_credito = ventas["TipoDocumento"].eq("NOTA_CREDITO")

ventas.loc[moneda_local, "ImporteGTQ"] = ventas.loc[
    moneda_local, "LineTotal"
]

ventas.loc[moneda_usd, "ImporteGTQ"] = (
    ventas.loc[moneda_usd, "LineTotal"]
    * ventas.loc[moneda_usd, "DocRate"]
)

# Notas de crédito expresadas en GTQ debido que credito no viene con DocCur
ventas.loc[es_credito, "ImporteGTQ"] = ventas.loc[
    es_credito, "LineTotal"
]

assert ventas["ImporteGTQ"].notna().all()

ventas["VentaNetaGTQ"] = ventas["ImporteGTQ"] * ventas["Signo"]

In [8]:
print("Filas:", len(ventas))
print("Venta neta GTQ:", ventas["VentaNetaGTQ"].sum())

Filas: 18739
Venta neta GTQ: 129870049.0762


In [9]:
ventas.head(10)

,TipoDocumento,DocEntry,LineNum,DocDate,DocSubType,ItemCode,WhsCode,U_Ubicacion,DocCur,DocRate,...,StockValue,GrssProfit,SKU,ItmsGrpCod,GrupoArticulo,WhsName,Ubicacion,Canal,ImporteGTQ,VentaNetaGTQ
0,FACTURA,555222,0,2026-08-01,--,4861,GT1CEDI,111.0,QTZ,1.0,...,3952.26,511.13,UN70U8000FPXPA,165.0,AUDIO Y VIDEO,CENTRO DE DISTRIBUCIÓN (CD),GT1CEDI-GT-001-UBI-FIS1-.COM-CENTRIX,RETAIL,4463.39,4463.39
1,FACTURA,555223,0,2026-08-01,--,3916,GT1COMER,50.0,QTZ,1.0,...,852.44,39.52,SM-R861NZSALTA,164.0,TELEFONIA Y ACC,PUNTOS DE VENTA OFFLINE,GT1COMER-GT-705-UBI-RETAIL,RETAIL,891.96,891.96
2,FACTURA,555224,0,2026-08-01,--,4617,GT1COMER,53.0,QTZ,1.0,...,6798.29,343.67,SM-S938BZTJGTO,164.0,TELEFONIA Y ACC,PUNTOS DE VENTA OFFLINE,GT1COMER-GT-708-UBI-RETAIL,RETAIL,7141.96,7141.96
3,FACTURA,555225,0,2026-08-01,--,4549,GT1COMER,50.0,QTZ,1.0,...,102.66,48.23,EF-GS938CBEGWW,164.0,TELEFONIA Y ACC,PUNTOS DE VENTA OFFLINE,GT1COMER-GT-705-UBI-RETAIL,RETAIL,150.89,150.89
4,FACTURA,555225,1,2026-08-01,--,5947,GT1COMER,50.0,QTZ,1.0,...,68.10,38.15,GP-FPR640HIBBW,164.0,TELEFONIA Y ACC,PUNTOS DE VENTA OFFLINE,GT1COMER-GT-705-UBI-RETAIL,RETAIL,106.25,106.25
5,FACTURA,555225,2,2026-08-01,--,2992,GT1COMER,50.0,QTZ,1.0,...,238.97,206.57,EP-T6530NBEGCA,164.0,TELEFONIA Y ACC,PUNTOS DE VENTA OFFLINE,GT1COMER-GT-705-UBI-RETAIL,RETAIL,445.54,445.54
6,FACTURA,555225,3,2026-08-01,--,2472,GT1COMER,50.0,QTZ,1.0,...,1900.38,-472.70,SM-R950NZSALTA,164.0,TELEFONIA Y ACC,PUNTOS DE VENTA OFFLINE,GT1COMER-GT-705-UBI-RETAIL,RETAIL,1427.68,1427.68
7,FACTURA,555226,0,2026-08-01,--,4183,GT1COMER,50.0,QTZ,1.0,...,878.09,13.87,SM-R630NZAALTA,164.0,TELEFONIA Y ACC,PUNTOS DE VENTA OFFLINE,GT1COMER-GT-705-UBI-RETAIL,RETAIL,891.96,891.96
8,FACTURA,555227,0,2026-08-01,--,6382,GT1COMER,33.0,QTZ,1.0,...,0.00,107.14,2383,171.0,SERVICIOS,PUNTOS DE VENTA OFFLINE,GT1COMER-GT-509-UBI-RETAIL,RETAIL,107.14,107.14
9,FACTURA,555228,0,2026-08-01,--,5754,GT1COMER,53.0,QTZ,1.0,...,2678.28,-0.60,QN55Q6FAAPXPA,165.0,AUDIO Y VIDEO,PUNTOS DE VENTA OFFLINE,GT1COMER-GT-708-UBI-RETAIL,RETAIL,2677.68,2677.68


## A2. ¿Cuál es la venta neta total del 1 al 21 de agosto, en GTQ y con dos decimales? Explica qué incluiste, qué excluiste y por qué.

In [10]:
print("Venta neta GTQ:", ventas["VentaNetaGTQ"].sum().round(2))

Venta neta GTQ: 129870049.08


La venta neta es Q129,870,049.08. Incluye los documentos no cancelados del 1 al 21 de agosto y resta las notas de crédito emitidas en ese período, incluso si la factura original no aparece en el extracto. Se excluyen los documentos con Canceled = Y o C.

Los importes en USD se convierten a GTQ. Las líneas con referencias faltantes se mantienen en el total financiero. El redondeo a dos decimales se realiza únicamente al presentar el resultado.


## A3. ¿Cuántos documentos de venta no deberían contarse? Indica cuántos son y cómo los identificaste.

In [11]:
documentos_excluidos = facturas_encabezado.loc[facturas_encabezado["Canceled"].isin(["Y", "C"])]
documentos_excluidos.groupby("Canceled").agg(documentos=("DocEntry", "nunique"))
documentos_excluidos["DocEntry"].nunique()

50

In [12]:
lineas_excluidas = facturas_detalle.merge(documentos_excluidos[["DocEntry", "Canceled"]], on="DocEntry", how="inner", validate="many_to_one")
lineas_excluidas.groupby("Canceled").size()

Canceled
C    39
Y    39
dtype: int64

Se excluyeron 25 documentos de venta cancelados (Y) y 25 documentos de cancelacion (C), estos 50 documentos corresponden a 78 lineas de facturas_detalle

## A4. ¿Cuántas líneas de factura necesitan un ajuste antes de sumarse con las demás? ¿Cuál es su monto correcto en GTQ?

In [13]:
facturas_encabezado["DocCur"].value_counts(dropna=False)

DocCur
QTZ    7086
USD     120
Name: count, dtype: int64

In [27]:
lineas_usd = ventas_facturas.loc[ventas_facturas["DocCur"].eq("USD")].copy()
lineas_usd["ImporteGTQ"] = (lineas_usd["LineTotal"] * lineas_usd["DocRate"])

print("Lineas USD: ", len(lineas_usd))
print("Importe original usd: USD", lineas_usd["LineTotal"].sum())
print("Importe correcto gtq: GTQ", lineas_usd["ImporteGTQ"].sum())

Lineas USD:  224
Importe original usd: USD 495986.57
Importe correcto gtq: GTQ 3799257.1262


## A5. ¿Cuántas filas de inventario_regional sobran? Explica qué criterio usaste para decidir cuál conservar.

In [15]:
inventario_diario.isna().sum()

DataMartDate            0
pais                    0
Canal                1346
Ubicacion               0
CodigoAlmacen           0
GrupoArticulo           0
SKU                    89
sum_Stock               0
sum_Comprometido        0
sum_Pedido              0
sum_CostoTotal        447
sum_CostoTotalUSD     447
dtype: int64

In [16]:
clave_inventario = [
    "DataMartDate",
    "pais",
    "Canal",
    "Ubicacion",
    "CodigoAlmacen",
    "GrupoArticulo",
    "SKU"
]

inventario_diario.duplicated(clave_inventario).sum()

np.int64(0)

In [17]:
clave_reducido = [
    "DataMartDate",
    "pais",
    "Canal",
    "Ubicacion",
    "CodigoAlmacen",
    "SKU"
]

inventario_diario.duplicated(clave_reducido).sum()

np.int64(246)

In [18]:
inventario_diario.loc[(inventario_diario["GrupoArticulo"]=="PROMOCIONALES") & (inventario_diario["sum_CostoTotal"] <= 0.5)]

,DataMartDate,pais,Canal,Ubicacion,CodigoAlmacen,GrupoArticulo,SKU,sum_Stock,sum_Comprometido,sum_Pedido,sum_CostoTotal,sum_CostoTotalUSD
493,2026-08-01,SLC-GT,B2B,GT1CEDI-GT-001-UBI-FIS1-B2B-CENTRIX,GT1CEDI,PROMOCIONALES,GF-P1757-1389,4,0,0,0.03,0.00
495,2026-08-01,SLC-GT,B2B,GT1CEDI-GT-001-UBI-FIS1-B2B-CENTRIX,GT1CEDI,PROMOCIONALES,Pich2L-PROM,4,0,0,0.04,0.01
496,2026-08-01,SLC-GT,B2B,GT1CEDI-GT-001-UBI-FIS1-B2B-CENTRIX,GT1CEDI,PROMOCIONALES,WMN4070TT/ZA,2,0,0,0.02,0.00
497,2026-08-01,SLC-GT,B2B,GT1CEDI-GT-001-UBI-FIS1-B2B-CENTRIX,GT1CEDI,PROMOCIONALES,WMN4277TT/ZX,1,0,0,0.01,0.00
563,2026-08-01,SLC-GT,DISTY,GT1CEDI-GT-001-UBI-FIS1- DISTY,GT1CEDI,PROMOCIONALES,BOLSALAV-1,1,0,0,0.01,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...
499857,2026-08-21,SLC-GT,RETAIL,GT1COMER-GT-903-UBI-RETAIL,GT1COMER,PROMOCIONALES,PODS,1,0,0,0.01,0.00
499860,2026-08-21,SLC-GT,RETAIL,GT1COMER-GT-903-UBI-RETAIL,GT1COMER,PROMOCIONALES,SIM-TIGOSPORTS,17,0,0,0.17,0.02
500214,2026-08-21,SLC-GT,RETAIL,GT1COMER-GT-904-UBI-RETAIL,GT1COMER,PROMOCIONALES,SIM-TIGOSPORTS,12,0,0,0.12,0.02
500647,2026-08-21,SLC-GT,SERVICE,GT1SERV-GT-001-UBI-SERVIFIS-PRODUCTO,GT1SERV,PROMOCIONALES,RCK-85PLG,4,0,0,0.04,0.01


Las filas con información faltante se conservan. Los canales sin identificar se reportan como desconocidos. Las filas sin SKU no se utilizan para análisis por producto, pero se mantienen en los totales de bodega. Los costos faltantes no se sustituyen por cero y se señalan como una limitación de la valoración.

No se eliminaron filas. No hay duplicados exactos ni claves repetidas al incluir fecha, país, canal, ubicación, bodega, grupo y SKU. Las 246 repeticiones de la clave reducida corresponden a grupos distintos; algunas además carecen de SKU. No existe evidencia suficiente para conservar una fila y descartar la otra.

Se conservaron las filas promocionales con costos bajos porque representan existencias registradas. Su valoración requiere cautela, pero no demuestra que sean filas sobrantes.

## A6. ¿Hay bodegas o días en los que el inventario no es confiable? Indica cuáles y cómo afectan tu cálculo de stock.

In [19]:
cobertura_inventario = pd.crosstab(inventario_diario["CodigoAlmacen"], inventario_diario["DataMartDate"])

with pd.option_context("display.max_columns", None):
    display(cobertura_inventario)

DataMartDate,2026-08-01,2026-08-02,2026-08-03,2026-08-04,2026-08-05,2026-08-06,2026-08-07,2026-08-08,2026-08-09,2026-08-10,2026-08-11,2026-08-12,2026-08-13,2026-08-14,2026-08-15,2026-08-16,2026-08-17,2026-08-18,2026-08-19,2026-08-20,2026-08-21
CodigoAlmacen,,,,,,,,,,,,,,,,,,,,,
GT1B2B,151,151,150,150,156,153,153,154,154,155,161,164,162,163,163,163,163,164,167,167,187
GT1CEDI,2061,2061,2049,2045,2046,2039,2044,2037,2037,2025,2017,2032,2023,2025,0,0,0,2005,1995,2010,2312
GT1COMER,18552,17946,17972,18001,18138,18146,18212,18170,18093,18028,18157,18282,18306,18368,18319,18244,18242,18275,18283,18213,18193
GT1CONTI,247,247,247,247,247,247,247,247,247,247,247,247,247,247,247,247,247,247,247,247,247
GT1DAÑO,260,260,262,266,266,266,408,409,409,415,450,449,449,449,449,449,449,451,451,462,463
GT1RECL,89,89,90,90,90,90,90,90,90,90,90,87,87,87,87,87,87,87,87,87,88
GT1SERV,1926,1926,1927,1938,1938,1941,1892,1890,1890,1909,1918,1929,1914,1918,1918,1918,1932,1959,1959,1959,1957
GT1TRAN,370,784,780,809,803,808,680,664,664,672,636,628,641,632,632,632,645,652,637,631,190
GT2CEDI,155,155,151,157,152,150,156,152,152,147,155,157,158,156,156,156,152,147,146,146,144


In [20]:
faltantes_cobertura = (
    cobertura_inventario
    .rename_axis(index="CodigoAlmacen", columns="Fecha")
    .stack()
    .rename("Filas")
    .reset_index()
)

faltantes_cobertura.loc[
    faltantes_cobertura["Filas"].eq(0),
    ["CodigoAlmacen", "Fecha"]
]

,CodigoAlmacen,Fecha
35,GT1CEDI,2026-08-15
36,GT1CEDI,2026-08-16
37,GT1CEDI,2026-08-17
189,GT2COMER,2026-08-01
190,GT2COMER,2026-08-02
191,GT2COMER,2026-08-03
192,GT2COMER,2026-08-04
193,GT2COMER,2026-08-05
194,GT2COMER,2026-08-06
195,GT2COMER,2026-08-07


In [21]:
resumen_diario = (
    inventario_diario
    .groupby(["CodigoAlmacen", "DataMartDate"])
    .agg(
        filas=("sum_Stock", "size"),
        stock_total=("sum_Stock", "sum"),
        skus_identificados=("SKU", "nunique"),
        costos_faltantes=("sum_CostoTotal", lambda s: s.isna().sum()),
    )
)

with pd.option_context("display.max_rows", None):
    display(
        resumen_diario.loc[["GT1CEDI", "GT2COMER", "GT1TRAN", "GT1DAÑO"]]
    )

filas  stock_total  skus_identificados  \
CodigoAlmacen DataMartDate                                           
GT1CEDI       2026-08-01     2061       217802                1322   
              2026-08-02     2061       217802                1322   
              2026-08-03     2049       206146                1314   
              2026-08-04     2045       198706                1312   
              2026-08-05     2046       168807                1312   
              2026-08-06     2039       167608                1310   
              2026-08-07     2044       166340                1307   
              2026-08-08     2037       166310                1307   
              2026-08-09     2037       166310                1307   
              2026-08-10     2025       162859                1298   
              2026-08-11     2017       156948                1291   
              2026-08-12     2032       151703                1307   
              2026-08-13     2023       151157                1298   
              2026-08-14     2025       150255                1297   
              2026-08-18     2005       152643                1302   
              2026-08-19     1995       150650                1299   
              2026-08-20     2010       149408                1312   
              2026-08-21     2312       153910                1488   
GT2COMER      2026-08-20        1           48                   1   
GT1TRAN       2026-08-01      370         5673                 355   
              2026-08-02      784         6687                 769   
              2026-08-03      780         6637                 765   
              2026-08-04      809         6609                 793   
              2026-08-05      803         5608                 788   
              2026-08-06      808         6830                 786   
              2026-08-07      680         4479                 672   
              2026-08-08      664         4316                 657   
              2026-08-09      664         4116                 657   
              2026-08-10      672         8474                 664   
              2026-08-11      636         6813                 635   
              2026-08-12      628         5152                 627   
              2026-08-13      641         5525                 640   
              2026-08-14      632         3514                 632   
              2026-08-15      632         3459                 632   
              2026-08-16      632         3459                 632   
              2026-08-17      645         3897                 645   
              2026-08-18      652         3757                 652   
              2026-08-19      637         3311                 637   
              2026-08-20      631         3734                 631   
              2026-08-21      190         2454                 190   
GT1DAÑO       2026-08-01      260          755                 247   
              2026-08-02      260          755                 247   
              2026-08-03      262          758                 248   
              2026-08-04      266          765                 251   
              2026-08-05      266          765                 251   
              2026-08-06      266          766                 251   
              2026-08-07      408         1017                 366   
              2026-08-08      409         1018                 367   
              2026-08-09      409         1018                 367   
              2026-08-10      415         1026                 373   
              2026-08-11      450         1094                 397   
              2026-08-12      449         1094                 397   
              2026-08-13      449         1094                 397   
              2026-08-14      449         1094                 397   
              2026-08-15      449         1094                 397   
              2026-08-16      449     

In [22]:
resumen_diario.loc[
    resumen_diario["costos_faltantes"].gt(0)
]

filas  stock_total  skus_identificados  \
CodigoAlmacen DataMartDate                                           
GT1COMER      2026-08-01    18552        61140                1433   
              2026-08-02    17946        59621                1430   
              2026-08-03    17972        59672                1435   
              2026-08-04    18001        59378                1438   
              2026-08-05    18138        58982                1449   
              2026-08-06    18146        58563                1448   
              2026-08-07    18212        58737                1452   
              2026-08-08    18170        58235                1451   
              2026-08-09    18093        57925                1451   
              2026-08-10    18028        57013                1450   
              2026-08-11    18157        58274                1451   
              2026-08-12    18282        58330                1451   
              2026-08-13    18306        58325                1447   
              2026-08-14    18368        58535                1447   
              2026-08-15    18319        58077                1447   
              2026-08-16    18244        57605                1445   
              2026-08-17    18242        57064                1446   
              2026-08-18    18275        56959                1445   
              2026-08-19    18283        57176                1445   
              2026-08-20    18213        56836                1443   
              2026-08-21    18193        56771                1442   
GT1SERV       2026-08-01     1926        14071                1898   
              2026-08-02     1926        14071                1898   
              2026-08-03     1927        14077                1899   
              2026-08-04     1938        14244                1910   
              2026-08-05     1938        14239                1910   
              2026-08-06     1941        14276                1913   
              2026-08-07     1892        13441                1864   
              2026-08-08     1890        13437                1862   
              2026-08-09     1890        13437                1862   
              2026-08-10     1909        13632                1881   
              2026-08-11     1918        13729                1890   
              2026-08-12     1929        13734                1901   
              2026-08-13     1914        13449                1886   
              2026-08-14     1918        13529                1890   
              2026-08-15     1918        13529                1890   
              2026-08-16     1918        13529                1890   
              2026-08-17     1932        13656                1904   
              2026-08-18     1959        13794                1931   
              2026-08-19     1959        13781                1931   
              2026-08-20     1959        13772                1931   
              2026-08-21     1957        13752                1929   

                            costos_faltantes  
CodigoAlmacen DataMartDate                    
GT1COMER      2026-08-01                  25  
              2026-08-02                  25  
              2026-08-03                  25  
              2026-08-04                  25  
              2026-08-05                  25  
              2026-08-06                  25  
              2026-08-07                  25  
              2026-08-08                  25  
              2026-08-09                  25  
              2026-08-10                  25  
              2026-08-11                  25  
              2026-08-12                  25  
              2026-08-13                  14  
              2026-08-14                  14  
              2026-08-15                  14  
              2026-08-16                  14  
              2026-08-17                  14  
              2026-08-18                  14  
       

Sí. GT1CEDI no tiene observaciones del 15 al 17 de agosto. GT2COMER solo registra una fila el 20 de agosto, por lo que no puede confirmarse su stock al corte del 21. Estas ausencias se consideran información desconocida y no stock cero; no se utilizan para inferir consumo o agotamiento.

GT1COMER presenta 25 filas sin costo por día del 1 al 12 de agosto y 14 por día del 13 al 21. GT1SERV presenta una fila sin costo cada día. En total son 447 observaciones con valoración incompleta.

Las cantidades registradas se conservan para calcular existencias, pero los totales monetarios se presentan como parciales. Los costos faltantes no se sustituyen por cero.

Se reportan cuatro bodegas con limitaciones de cobertura o valoración: GT1CEDI, GT2COMER, GT1COMER y GT1SERV. Esto no significa que las cantidades de las cuatro bodegas sean incorrectas.


## A7. ¿Qué monto de notas de crédito corresponde a facturas emitidas antes de agosto? ¿Deberían restarse de la venta de agosto? Justifica.

In [23]:
creditos_sin_factura = ventas_credito.loc[~ventas_credito["BaseEntry"].isin(facturas_encabezado["DocEntry"])].copy()

print("Líneas:", len(creditos_sin_factura))
print("Notas de crédito:", creditos_sin_factura["DocEntry"].nunique())
print("Facturas referenciadas:", creditos_sin_factura["BaseEntry"].nunique())
print("Monto GTQ:", creditos_sin_factura["LineTotal"].sum())

Líneas: 70
Notas de crédito: 66
Facturas referenciadas: 69
Monto GTQ: 2322769.28


Se considera que las referencias ausentes corresponden a facturas anteriores a agosto, dado que el extracto de facturas cubre únicamente del 1 al 21. Este supuesto no puede verificarse sin los encabezados históricos.

Sí se restan de la venta neta de agosto porque las notas de crédito fueron emitidas en ese período. El criterio utilizado reconoce las facturas y notas de crédito según su propia fecha de emisión, independientemente de la fecha de la factura original.

## A8. Si haces el join con el catálogo sin revisar nada, ¿cuánto cambia la venta neta? ¿Por qué?

In [26]:
ventas_antes_join = ventas[["TipoDocumento", "DocEntry", "LineNum", "ItemCode", "VentaNetaGTQ"]].copy()

ventas_catalogo_original = ventas_antes_join.merge(catalogo_productos, on="ItemCode", how="left")

total_revisado = ventas_antes_join["VentaNetaGTQ"].sum()
total_sin_revision = ventas_catalogo_original["VentaNetaGTQ"].sum()

print("Filas revisadas:", len(ventas_antes_join))
print("Filas sin revisar:", len(ventas_catalogo_original))
print("Venta neta revisada:", total_revisado)
print("Venta neta sin revisar:", total_sin_revision)
print("Cambio neta GTQ:", total_sin_revision - total_revisado)

Filas revisadas: 18739
Filas sin revisar: 18779
Venta neta revisada: 129870049.0762
Venta neta sin revisar: 198117144.81179997
Cambio neta GTQ: 68247095.73559998


Al unir las ventas con el catálogo original, las filas aumentan de 18,739 a 18,779. Los ItemCode repetidos generan coincidencias adicionales y hacen que algunos importes se contabilicen dos veces.

La venta neta pasa de Q129,870,049.0762 a Q198,117,144.8118, un aumento aproximado de Q68,247,095.7356. Para evitarlo, se utiliza una sola fila por ItemCode en el catálogo revisado.

In [25]:
codigos_repetidos = catalogo_productos.loc[catalogo_productos.duplicated("ItemCode", keep=False), "ItemCode"].unique()

ventas_afectadas = ventas_antes_join.loc[ventas_antes_join["ItemCode"].isin(codigos_repetidos)]

display(ventas_afectadas)

,TipoDocumento,DocEntry,LineNum,ItemCode,VentaNetaGTQ
1276,FACTURA,556059,0,6862,4.964410e+06
1322,FACTURA,556084,0,6860,4.825443e+06
1479,FACTURA,556159,0,4722,2.959904e+06
1480,FACTURA,556160,0,4722,1.606805e+06
2406,FACTURA,556581,0,4371,9.388579e+05
2615,FACTURA,556622,0,6416,1.280261e+06
2725,FACTURA,556657,0,6686,3.444511e+06
2758,FACTURA,556673,0,5656,1.082125e+04
2759,FACTURA,556673,1,5656,1.406763e+06
2760,FACTURA,556673,2,5656,1.298550e+06
